# 03 - Opener Statistics

Analyze first, second, and third move patterns from simulated games.

This notebook is designed for runs where `RANDOM_FIRST_MOVE=True` in `01_simulate_games.ipynb`.

In [ ]:
import os
import sqlite3
import pandas as pd

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)

In [ ]:
# ---- Config ----
DB_PATH = os.path.join('data', 'games.sqlite')
TOP_N = 3
MIN_SAMPLE_FOR_BEST_REACTION = 20

print(f'DB: {DB_PATH}')
print(f'TOP_N: {TOP_N}')
print(f'MIN_SAMPLE_FOR_BEST_REACTION: {MIN_SAMPLE_FOR_BEST_REACTION}')

In [ ]:
# ---- Load data ----
conn = sqlite3.connect(DB_PATH)

games_df = pd.read_sql(
    '''
    SELECT game_id, winner, total_moves, white_depth, black_depth, termination
    FROM games
    ''',
    conn,
)

moves_df = pd.read_sql(
    '''
    SELECT
        game_id,
        move_number,
        color,
        from_vertex || '->' || to_vertex AS move
    FROM moves
    WHERE move_number <= 3
    ORDER BY game_id, move_number
    ''',
    conn,
)

conn.close()

m1 = moves_df[moves_df['move_number'] == 1][['game_id', 'move']].rename(columns={'move': 'move_1'})
m2 = moves_df[moves_df['move_number'] == 2][['game_id', 'move']].rename(columns={'move': 'move_2'})
m3 = moves_df[moves_df['move_number'] == 3][['game_id', 'move']].rename(columns={'move': 'move_3'})

opening_df = (
    games_df
    .merge(m1, on='game_id', how='inner')
    .merge(m2, on='game_id', how='left')
    .merge(m3, on='game_id', how='left')
)

opening_df['white_win'] = (opening_df['winner'] == 'WHITE').astype(int)
opening_df['black_win'] = (opening_df['winner'] == 'BLACK').astype(int)
opening_df['draw'] = (opening_df['winner'] == 'DRAW').astype(int)

print(f'Games loaded: {len(games_df):,}')
print(f'Games with move_1: {len(opening_df):,}')
opening_df.head()

In [ ]:
def with_percentages(df, rate_cols):
    out = df.copy()
    for col in rate_cols:
        if col in out.columns:
            out[col] = (out[col] * 100).round(2)
    return out

## Top opening moves (Move 1)

In [ ]:
first_stats = (
    opening_df
    .groupby('move_1', as_index=False)
    .agg(
        games=('game_id', 'count'),
        white_win_rate=('white_win', 'mean'),
        black_win_rate=('black_win', 'mean'),
        draw_rate=('draw', 'mean'),
        avg_total_moves=('total_moves', 'mean'),
    )
    .sort_values(['games', 'white_win_rate'], ascending=[False, False])
)

top_openers = first_stats.head(TOP_N)
display(with_percentages(top_openers, ['white_win_rate', 'black_win_rate', 'draw_rate']))

## Best responses by opener (Move 2 conditioned on Move 1)

In [ ]:
response_stats = (
    opening_df
    .dropna(subset=['move_2'])
    .groupby(['move_1', 'move_2'], as_index=False)
    .agg(
        games=('game_id', 'count'),
        black_win_rate=('black_win', 'mean'),
        white_win_rate=('white_win', 'mean'),
        draw_rate=('draw', 'mean'),
        avg_total_moves=('total_moves', 'mean'),
    )
)

for opener in top_openers['move_1'].tolist():
    subset = (
        response_stats[response_stats['move_1'] == opener]
        .sort_values(['games', 'black_win_rate'], ascending=[False, False])
        .head(TOP_N)
    )
    print(f'Opener: {opener}')
    display(with_percentages(subset, ['black_win_rate', 'white_win_rate', 'draw_rate']))

    best_reaction = (
        response_stats[
            (response_stats['move_1'] == opener)
            & (response_stats['games'] >= MIN_SAMPLE_FOR_BEST_REACTION)
        ]
        .sort_values(['black_win_rate', 'games'], ascending=[False, False])
        .head(1)
    )
    if len(best_reaction) == 0:
        print('  No best-reaction row at the sample threshold.\n')
    else:
        print('  Best reaction by BLACK win rate (sample-filtered):')
        display(with_percentages(best_reaction, ['black_win_rate', 'white_win_rate', 'draw_rate']))

## Third moves after opener + response (Move 3 conditioned on Moves 1-2)

In [ ]:
third_stats = (
    opening_df
    .dropna(subset=['move_2', 'move_3'])
    .groupby(['move_1', 'move_2', 'move_3'], as_index=False)
    .agg(
        games=('game_id', 'count'),
        white_win_rate=('white_win', 'mean'),
        black_win_rate=('black_win', 'mean'),
        draw_rate=('draw', 'mean'),
        avg_total_moves=('total_moves', 'mean'),
    )
)

for opener in top_openers['move_1'].tolist():
    print(f'Opener: {opener}')
    top_responses = (
        response_stats[response_stats['move_1'] == opener]
        .sort_values(['games', 'black_win_rate'], ascending=[False, False])
        .head(TOP_N)
    )

    for response in top_responses['move_2'].tolist():
        followups = (
            third_stats[
                (third_stats['move_1'] == opener)
                & (third_stats['move_2'] == response)
            ]
            .sort_values(['games', 'white_win_rate'], ascending=[False, False])
            .head(TOP_N)
        )
        print(f'  Response: {response}')
        display(with_percentages(followups, ['white_win_rate', 'black_win_rate', 'draw_rate']))

## Optional exports

Uncomment if you want reusable CSV artifacts.

In [ ]:
# first_stats.to_csv('data/opener_move1_stats.csv', index=False)
# response_stats.to_csv('data/opener_move2_response_stats.csv', index=False)
# third_stats.to_csv('data/opener_move3_followup_stats.csv', index=False)